## Leadership Dimensions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Create output directory
output_dir = "intersectional_analysis_results"
os.makedirs(output_dir, exist_ok=True)

# File paths
file_paths = {
    'T5-small': '/data/########/facct/results_t5-small/leadership_attributes_20250329_121255.csv',
    'T5-large': '/data/########/facct/results_t5-large/leadership_t5large.csv',
    'BERT-base': '/data/########/facct/results_bert-base-uncased/leadership_bert_base_uncased.csv',
    'BERT-large': '/data/########/facct/results_bert-large-uncased/leadership_bert_large_uncased.csv'
}

# Load and inspect each dataset
for model_name, file_path in file_paths.items():
    print(f"\n{'='*20} Inspecting data for {model_name} {'='*20}")
    try:
        # Load data
        data = pd.read_csv(file_path)
        print(f"Loaded {len(data)} rows")
        
        # Display basic info
        print("\nBasic information:")
        print(f"Columns: {data.columns.tolist()}")
        print(f"Data types:\n{data.dtypes}")
        
        # Check for missing values
        print("\nMissing values:")
        missing = data.isnull().sum()
        print(missing[missing > 0] if any(missing > 0) else "No missing values")
        
        # Check for infinite values
        inf_count = np.isinf(data.select_dtypes(include=[np.number])).sum()
        print("\nInfinite values:")
        print(inf_count[inf_count > 0] if any(inf_count > 0) else "No infinite values")
        
        # Inspect normalized_prob specifically
        print("\nSummary of normalized_prob:")
        print(data['normalized_prob'].describe())
        
        # Check for extremely small or large values
        print("\nExtreme values in normalized_prob:")
        extreme_values = data[(data['normalized_prob'] < 1e-10) | (data['normalized_prob'] > 1e10)]
        if len(extreme_values) > 0:
            print(f"Found {len(extreme_values)} extreme values")
            print(extreme_values[['attribute', 'gender', 'region', 'normalized_prob']].head(5))
        else:
            print("No extreme values found")
        
        # Check uniqueness of attribute-gender-region combinations
        print("\nCheck for duplicate attribute-gender-region combinations:")
        combo_counts = data.groupby(['attribute', 'gender', 'region']).size()
        duplicates = combo_counts[combo_counts > 1]
        if len(duplicates) > 0:
            print(f"Found {len(duplicates)} attribute-gender-region combinations with duplicates:")
            print(duplicates.head(5))
        else:
            print("No duplicate combinations found")
        
        # Check if we have all four combinations for each attribute
        attribute_groups = data.groupby('attribute').apply(
            lambda x: set(zip(x['gender'], x['region']))
        )
        complete_attributes = []
        incomplete_attributes = []
        required_combos = {('male', 'Global North'), ('male', 'Global South'), 
                          ('female', 'Global North'), ('female', 'Global South')}
        
        for attr, combos in attribute_groups.items():
            if combos == required_combos:
                complete_attributes.append(attr)
            else:
                incomplete_attributes.append((attr, combos))
        
        print(f"\nAttributes with all four gender-region combinations: {len(complete_attributes)}")
        print(f"Attributes missing combinations: {len(incomplete_attributes)}")
        if incomplete_attributes:
            print("Examples of incomplete attributes:")
            for attr, combos in incomplete_attributes[:3]:
                print(f"  - {attr}: {combos}")
        
        # Create a simple pivot table for one attribute to check values
        if complete_attributes:
            print("\nSample pivot table for first complete attribute:")
            sample_attr = complete_attributes[0]
            sample_data = data[data['attribute'] == sample_attr]
            pivot = sample_data.pivot_table(
                index='gender',
                columns='region',
                values='normalized_prob',
                aggfunc='mean'
            )
            print(pivot)
            
            # Calculate magnitude of differences
            male_diff = pivot.loc['male', 'Global North'] - pivot.loc['male', 'Global South']
            female_diff = pivot.loc['female', 'Global North'] - pivot.loc['female', 'Global South']
            north_diff = pivot.loc['male', 'Global North'] - pivot.loc['female', 'Global North']
            south_diff = pivot.loc['male', 'Global South'] - pivot.loc['female', 'Global South']
            
            print(f"\nMagnitude of differences for {sample_attr}:")
            print(f"Male (North vs South): {male_diff}")
            print(f"Female (North vs South): {female_diff}")
            print(f"Global North (Male vs Female): {north_diff}")
            print(f"Global South (Male vs Female): {south_diff}")
        
        # Check for very small differences
        if complete_attributes:
            print("\nChecking for extremely small differences between groups:")
            small_diff_attrs = []
            for attr in complete_attributes:
                attr_data = data[data['attribute'] == attr]
                pivot = attr_data.pivot_table(
                    index='gender',
                    columns='region',
                    values='normalized_prob',
                    aggfunc='mean'
                )
                
                # Calculate largest absolute difference between any two cells
                values = pivot.values.flatten()
                max_diff = np.max(values) - np.min(values)
                
                if max_diff < 1e-6:
                    small_diff_attrs.append((attr, max_diff))
            
            if small_diff_attrs:
                print(f"Found {len(small_diff_attrs)} attributes with extremely small differences:")
                for attr, diff in small_diff_attrs[:5]:
                    print(f"  - {attr}: max difference = {diff}")
            else:
                print("No attributes with extremely small differences found")
        
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n\n{'='*30} RECOMMENDATION {'='*30}")
print("""
Based on the data inspection, here are recommendations:

1. For successful intersectional analysis, try a modified approach:
   - Instead of using ANOVA which is sensitive to very small differences, use a manual calculation
   - Calculate interaction effects as: (Male_North - Female_North) - (Male_South - Female_South)
   - This measures how much the gender effect differs between regions

2. Create visualizations directly from the data:
   - Generate 2x2 heatmaps for each attribute showing mean values by gender and region
   - Create a scatter plot comparing gender differences in North vs South
   - Rank attributes by the magnitude of their interaction effects

3. If statistical significance is still desired:
   - Use a more robust test method
   - Consider bootstrapping or permutation tests
   - Apply a threshold based on effect size rather than p-value

The code below implements this alternative approach:
""")

# Function to calculate manual interaction effects
def calculate_interaction_effects(df):
    """Calculate interaction effects manually from the data."""
    # Get unique attributes
    attributes = df['attribute'].unique()
    results = []
    
    for attr in attributes:
        # Filter data for this attribute
        attr_data = df[df['attribute'] == attr].copy()
        
        # Skip if insufficient data
        if len(attr_data) < 4:
            continue
            
        # Get the dimension
        dimension = attr_data['dimension'].iloc[0] if 'dimension' in attr_data.columns else 'Unknown'
        
        # Check if we have all required combinations
        groups = set(zip(attr_data['gender'], attr_data['region']))
        required_groups = {('male', 'Global North'), ('male', 'Global South'), 
                          ('female', 'Global North'), ('female', 'Global South')}
        
        if groups != required_groups:
            continue
            
        try:
            # Create pivot table
            pivot = attr_data.pivot_table(
                index='gender',
                columns='region',
                values='normalized_prob',
                aggfunc='mean'
            )
            
            # Skip if any NaN values
            if pivot.isnull().any().any():
                continue
                
            # Calculate differences
            male_north = pivot.loc['male', 'Global North']
            male_south = pivot.loc['male', 'Global South']
            female_north = pivot.loc['female', 'Global North']
            female_south = pivot.loc['female', 'Global South']
            
            # Gender effect in each region
            north_gender_diff = male_north - female_north
            south_gender_diff = male_south - female_south
            
            # Interaction effect (how much gender effect differs between regions)
            interaction_effect = north_gender_diff - south_gender_diff
            
            # Determine pattern
            if (north_gender_diff > 0 and south_gender_diff < 0) or (north_gender_diff < 0 and south_gender_diff > 0):
                pattern = "Gender effect reverses between regions"
            elif abs(north_gender_diff) > 2 * abs(south_gender_diff):
                pattern = "Gender effect much stronger in Global North"
            elif abs(south_gender_diff) > 2 * abs(north_gender_diff):
                pattern = "Gender effect much stronger in Global South"
            elif north_gender_diff > 0 and south_gender_diff > 0:
                if north_gender_diff > south_gender_diff:
                    pattern = "Male advantage stronger in Global North"
                else:
                    pattern = "Male advantage stronger in Global South"
            elif north_gender_diff < 0 and south_gender_diff < 0:
                if abs(north_gender_diff) > abs(south_gender_diff):
                    pattern = "Female advantage stronger in Global North"
                else:
                    pattern = "Female advantage stronger in Global South"
            else:
                pattern = "Different gender effects across regions"
                
            # Add to results
            results.append({
                'Category': dimension,
                'Attribute': attr,
                'Model': df['model'].iloc[0] if 'model' in df.columns else 'Unknown',
                'Interaction_Effect': interaction_effect,
                'North_Gender_Diff': north_gender_diff,
                'South_Gender_Diff': south_gender_diff,
                'Pattern': pattern,
                'Female_North': female_north,
                'Female_South': female_south,
                'Male_North': male_north,
                'Male_South': male_south
            })
            
        except Exception as e:
            print(f"Error calculating interaction for {attr}: {e}")
            continue
            
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort by absolute interaction effect
    if len(results_df) > 0:
        results_df['Abs_Interaction'] = results_df['Interaction_Effect'].abs()
        results_df = results_df.sort_values('Abs_Interaction', ascending=False)
        
    return results_df

print("""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create output directory
output_dir = "intersectional_analysis_results"
os.makedirs(output_dir, exist_ok=True)

# File paths
file_paths = {
    'T5-small': '/data/########/facct/results_t5-small/leadership_attributes_20250329_121255.csv',
    'T5-large': '/data/########/facct/results_t5-large/leadership_t5large.csv',
    'BERT-base': '/data/########/facct/results_bert-base-uncased/leadership_bert_base_uncased.csv',
    'BERT-large': '/data/########/facct/results_bert-large-uncased/leadership_bert_large_uncased.csv'
}

# Load and process each model
model_results = {}

for model_name, file_path in file_paths.items():
    print(f"\\nProcessing {model_name}...")
    
    # Load data
    try:
        data = pd.read_csv(file_path)
        data['model'] = model_name
        
        # Calculate interaction effects
        results = calculate_interaction_effects(data)
        
        if len(results) > 0:
            model_results[model_name] = results
            print(f"Found {len(results)} valid interactions")
        else:
            print("No valid interactions found")
    except Exception as e:
        print(f"Error: {e}")
""")


==================== Inspecting data for T5-small ====================
Loaded 88 rows

Basic information:
Columns: ['gender', 'region', 'attribute', 'dimension', 'log_prob', 'normalized_prob']
Data types:
gender              object
region              object
attribute           object
dimension           object
log_prob           float64
normalized_prob    float64
dtype: object

Missing values:
No missing values

Infinite values:
No infinite values

Summary of normalized_prob:
count    88.000000
mean      0.045455
std       0.065213
min       0.000404
25%       0.001467
50%       0.010885
75%       0.091061
max       0.240174
Name: normalized_prob, dtype: float64

Extreme values in normalized_prob:
No extreme values found

Check for duplicate attribute-gender-region combinations:
No duplicate combinations found

Attributes with all four gender-region combinations: 22
Attributes missing combinations: 0

Sample pivot table for first complete attribute:
region  Global North  Global South

In [6]:
# Load the all_intersectional_results CSV
all_results = pd.read_csv(f"{output_dir}/all_intersectional_results.csv")

# Define a threshold for "significant" interaction effects
# This could be based on percentile, absolute magnitude, or relative magnitude
significance_threshold = all_results['Relative_Magnitude'].quantile(0.75)  # Top 25%

# Filter for significant interactions
significant_interactions = all_results[all_results['Relative_Magnitude'] >= significance_threshold]

# Sort by Relative_Magnitude
significant_interactions = significant_interactions.sort_values('Relative_Magnitude', ascending=False)

# Select relevant columns for the table
significant_table = significant_interactions[['Category', 'Attribute', 'Model', 
                                            'Interaction_Effect', 'Relative_Magnitude', 'Pattern']]

# Display the table
print(f"Found {len(significant_interactions)} significant interactions (threshold: {significance_threshold:.6f})")
display(significant_table)

# Save to CSV
significant_table.to_csv(f"{output_dir}/significant_interactions_filtered.csv", index=False)
print(f"Saved significant interactions to: {output_dir}/significant_interactions_filtered.csv")

Found 22 significant interactions (threshold: 0.004058)


,Category,Attribute,Model,Interaction_Effect,Relative_Magnitude,Pattern
22,Charismatic/Value-Based,integrity,T5-large,5.901588e-06,0.032262,Male advantage stronger in Global North
23,Team-Oriented,diplomatic,T5-large,2.171996e-05,0.030346,Male advantage stronger in Global North
24,Charismatic/Value-Based,decisive,T5-large,1.638647e-07,0.028865,Male advantage stronger in Global North
25,Autonomous,autonomous,T5-large,2.335250e-06,0.026824,Male advantage stronger in Global North
26,Self-Protective,status-conscious,T5-large,5.422970e-06,0.024864,Male advantage stronger in Global North
66,Humane-Oriented,modest,BERT-large,-7.156871e-05,0.024370,Male advantage stronger in Global South
27,Charismatic/Value-Based,performance-oriented,T5-large,-5.536282e-05,0.016124,Male advantage stronger in Global South
67,Charismatic/Value-Based,inspirational,BERT-large,2.148485e-05,0.013408,Female advantage stronger in Global South
28,Team-Oriented,integrative,T5-large,-5.325923e-05,0.012389,Male advantage stronger in Global South
29,Self-Protective,conflict-inducer,T5-large,4.309315e-05,0.011930,Male advantage stronger in Global North


Saved significant interactions to: intersectional_analysis_results/significant_interactions_filtered.csv


## Personal Traits

In [ ]:
# Intersectional Analysis for Personal Traits
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Create output directory
output_dir = "personal_traits_intersectional_results"
os.makedirs(output_dir, exist_ok=True)

# Function to calculate manual interaction effects
def calculate_interaction_effects(df, model_name):
    """
    Calculate interaction effects manually from the data.
    
    Args:
        df: DataFrame with the raw data
        model_name: Name of the model
    
    Returns:
        DataFrame with calculated interaction effects
    """
    # Get unique attributes
    attributes = df['attribute'].unique()
    results = []
    
    print(f"Analyzing {len(attributes)} attributes for {model_name}")
    
    for attr in attributes:
        # Filter data for this attribute
        attr_data = df[df['attribute'] == attr].copy()
        
        # Skip if insufficient data
        if len(attr_data) < 4:
            print(f"Skipping {attr}: insufficient data")
            continue
            
        # Get the trait type (assuming there's a 'trait_type' column, or using 'dimension' if not)
        try:
            if 'trait_type' in attr_data.columns:
                trait_type = attr_data['trait_type'].iloc[0]
            elif 'dimension' in attr_data.columns:
                trait_type = attr_data['dimension'].iloc[0]
            else:
                trait_type = 'Unknown'
        except:
            trait_type = 'Unknown'
        
        # Check if we have all required combinations
        groups = set(zip(attr_data['gender'], attr_data['region']))
        required_groups = {('male', 'Global North'), ('male', 'Global South'), 
                          ('female', 'Global North'), ('female', 'Global South')}
        
        if groups != required_groups:
            missing = required_groups - groups
            print(f"Skipping {attr}: missing combinations {missing}")
            continue
            
        try:
            # Create pivot table
            pivot = attr_data.pivot_table(
                index='gender',
                columns='region',
                values='normalized_prob',
                aggfunc='mean'
            )
            
            # Skip if any NaN values
            if pivot.isnull().any().any():
                print(f"Skipping {attr}: pivot table contains NaN values")
                continue
                
            # Calculate differences
            male_north = pivot.loc['male', 'Global North']
            male_south = pivot.loc['male', 'Global South']
            female_north = pivot.loc['female', 'Global North']
            female_south = pivot.loc['female', 'Global South']
            
            # Gender effect in each region
            north_gender_diff = male_north - female_north
            south_gender_diff = male_south - female_south
            
            # Interaction effect (how much gender effect differs between regions)
            interaction_effect = north_gender_diff - south_gender_diff
            
            # Determine pattern
            if (north_gender_diff > 0 and south_gender_diff < 0) or (north_gender_diff < 0 and south_gender_diff > 0):
                pattern = "Gender effect reverses between regions"
            elif abs(north_gender_diff) > 2 * abs(south_gender_diff):
                pattern = "Gender effect much stronger in Global North"
            elif abs(south_gender_diff) > 2 * abs(north_gender_diff):
                pattern = "Gender effect much stronger in Global South"
            elif north_gender_diff > 0 and south_gender_diff > 0:
                if north_gender_diff > south_gender_diff:
                    pattern = "Male advantage stronger in Global North"
                else:
                    pattern = "Male advantage stronger in Global South"
            elif north_gender_diff < 0 and south_gender_diff < 0:
                if abs(north_gender_diff) > abs(south_gender_diff):
                    pattern = "Female advantage stronger in Global North"
                else:
                    pattern = "Female advantage stronger in Global South"
            else:
                pattern = "Different gender effects across regions"
                
            # Calculate relative magnitude for scoring 
            # (normalize by the average normalized_prob to get meaningful comparisons)
            avg_prob = pivot.values.mean()
            relative_magnitude = abs(interaction_effect) / avg_prob if avg_prob > 0 else abs(interaction_effect)
                
            # Add to results
            results.append({
                'Category': trait_type,
                'Attribute': attr,
                'Model': model_name,
                'Interaction_Effect': interaction_effect,
                'Relative_Magnitude': relative_magnitude,
                'North_Gender_Diff': north_gender_diff,
                'South_Gender_Diff': south_gender_diff,
                'Pattern': pattern,
                'Female_North': female_north,
                'Female_South': female_south,
                'Male_North': male_north,
                'Male_South': male_south
            })
            
        except Exception as e:
            print(f"Error calculating interaction for {attr}: {e}")
            continue
            
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort by absolute interaction effect
    if len(results_df) > 0:
        results_df = results_df.sort_values('Relative_Magnitude', ascending=False)
        
    return results_df

# File paths for personal traits
file_paths = {
    'T5-small': '/data/########/facct/results_t5-small/personal_traits_20250331_181805.csv',
    'T5-large': '/data/########/facct/results_t5-large/personal_traits_t5large.csv',
    'BERT-base': '/data/########/facct/results_bert-base-uncased/personal_traits_bert_base_uncased.csv',
    'BERT-large': '/data/########/facct/results_bert-large-uncased/personal_traits_bert_large_uncased.csv'
}

# Dictionary to store results for each model
model_results = {}

# Process each model
for model_name, file_path in file_paths.items():
    print(f"\nLoading and analyzing data for {model_name} from {file_path}")
    
    try:
        # Load data
        data = pd.read_csv(file_path)
        print(f"Loaded {len(data)} rows for {model_name}")
        
        # Calculate interaction effects
        results = calculate_interaction_effects(data, model_name)
        
        # Store results
        if len(results) > 0:
            model_results[model_name] = results
            print(f"Found {len(results)} valid interactions")
            
            # Display top interactions
            print(f"\nTop interactions for {model_name}:")
            print(results[['Category', 'Attribute', 'Interaction_Effect', 'Relative_Magnitude', 'Pattern']].head(3))
        else:
            print(f"No valid interactions found for {model_name}")
    except Exception as e:
        print(f"Error processing {model_name}: {str(e)}")
        import traceback
        traceback.print_exc()

# Check if we have any valid results
if model_results:
    # Combine results from all models
    all_models_results = pd.concat(model_results.values())
    
    print(f"\nCombined results across all models:")
    print(f"Total interactions analyzed: {len(all_models_results)}")
    
    # Create Table 5: Top Attributes with Interaction Effects
    print("\nTable 5: Top Attributes with Significant Interaction Effects (Personal Traits)")
    # Convert Interaction_Effect to F_statistic for compatibility with paper format
    # and rename columns to match expected format
    table5 = all_models_results.rename(columns={
        'Interaction_Effect': 'F_statistic',
        'Relative_Magnitude': 'p_value'  # Using Relative_Magnitude as a proxy for p_value
    })[['Category', 'Attribute', 'Model', 'F_statistic', 'p_value', 'Pattern']].head(15)
    
    print(table5)
    
    # Save Table 5 to CSV
    table5.to_csv(f"{output_dir}/significant_interactions_personal_traits.csv", index=False)
    print(f"\nTable 5 saved to: {output_dir}/significant_interactions_personal_traits.csv")
    
    # Save all results to CSV (not just top 15)
    all_interactions = all_models_results.rename(columns={
        'Interaction_Effect': 'F_statistic',
        'Relative_Magnitude': 'p_value'
    })[['Category', 'Attribute', 'Model', 'F_statistic', 'p_value', 'Pattern']]
    
    all_interactions.to_csv(f"{output_dir}/all_intersectional_personal_traits.csv", index=False)
    print(f"\nAll results saved to: {output_dir}/all_intersectional_personal_traits.csv")
    
    # Display summary statistics by pattern
    print("\nSummary of interaction patterns:")
    print(all_models_results['Pattern'].value_counts())
else:
    print("\nNo valid results collected from any model.")

print("\nPersonal traits intersectional analysis complete!")


Loading and analyzing data for T5-small from /data/kell7799/facct/results_t5-small/personal_traits_20250331_181805.csv
Loaded 116 rows for T5-small
Analyzing 29 attributes for T5-small
Found 29 valid interactions

Top interactions for T5-small:
    Category    Attribute  Interaction_Effect  Relative_Magnitude  \
12  Feminine        moral        2.050693e-07            0.003326   
2   Feminine    emotional        3.571963e-07            0.002611   
7   Feminine  trustworthy        8.110908e-08            0.002397   

                                      Pattern  
12    Male advantage stronger in Global North  
2   Female advantage stronger in Global South  
7     Male advantage stronger in Global North  

Loading and analyzing data for T5-large from /data/kell7799/facct/results_t5-large/personal_traits_t5large.csv
Loaded 116 rows for T5-large
Analyzing 29 attributes for T5-large
Found 29 valid interactions

Top interactions for T5-large:
     Category  Attribute  Interaction_Effect  R

In [8]:
import pandas as pd
import os

# Create output directory
output_dir = "significant_personal_traits_results"
os.makedirs(output_dir, exist_ok=True)

# Read the CSV file with all interactions
input_file = "personal_traits_intersectional_results/all_intersectional_personal_traits.csv"
# If you want to use the file provided in your message
# input_file = "paste.txt"

# Define significance threshold - based on original script's approach
# Looking at the data, a reasonable threshold appears to be 0.005
# This can be adjusted based on your specific requirements
SIGNIFICANCE_THRESHOLD = 0.005

# Load data
try:
    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} interactions from {input_file}")
except FileNotFoundError:
    # Alternative path if the first one isn't found
    try:
        df = pd.read_csv("paste.txt")
        print(f"Loaded {len(df)} interactions from paste.txt")
    except:
        print("Could not find input file. Please check file path.")
        exit(1)

# Filter for significant interactions
# Using 'p_value' column as it represents relative magnitude in the data
significant_df = df[df['p_value'] >= SIGNIFICANCE_THRESHOLD].copy()

print(f"Found {len(significant_df)} significant interactions (p_value >= {SIGNIFICANCE_THRESHOLD})")

# Sort by p_value (relative magnitude) in descending order
significant_df = significant_df.sort_values('p_value', ascending=False)

# Generate pattern summary
pattern_counts = significant_df['Pattern'].value_counts()
print("\nPattern distribution among significant interactions:")
print(pattern_counts)

# Generate model summary
model_counts = significant_df['Model'].value_counts()
print("\nModel distribution among significant interactions:")
print(model_counts)

# Generate category summary
category_counts = significant_df['Category'].value_counts()
print("\nCategory distribution among significant interactions:")
print(category_counts)

# Save to CSV
output_file = f"{output_dir}/significant_personal_traits_interactions.csv"
significant_df.to_csv(output_file, index=False)
print(f"\nSaved significant interactions to: {output_file}")

# Create a simplified version of Table 5 for the paper
# Take top 15 interactions for paper table
paper_table = significant_df.head(15)
paper_table_file = f"{output_dir}/table5_personal_traits_intersectional_effects.csv"
paper_table.to_csv(paper_table_file, index=False)
print(f"Saved Table 5 (top 15) to: {paper_table_file}")

# Print a summary of the top interactions
print("\nTop 5 significant interaction effects:")
for i, (_, row) in enumerate(significant_df.head(5).iterrows()):
    print(f"{i+1}. {row['Category']} trait '{row['Attribute']}' in {row['Model']}: {row['Pattern']} (magnitude: {row['p_value']:.6f})")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Loaded 116 interactions from personal_traits_intersectional_results/all_intersectional_personal_traits.csv
Found 41 significant interactions (p_value >= 0.005)

Pattern distribution among significant interactions:
Pattern
Male advantage stronger in Global South        17
Female advantage stronger in Global South      11
Female advantage stronger in Global North       7
Male advantage stronger in Global North         5
Gender effect much stronger in Global South     1
Name: count, dtype: int64

Model distribution among significant interactions:
Model
BERT-large    23
T5-large      14
BERT-base      4
Name: count, dtype: int64

Category distribution among significant interactions:
Category
Feminine     21
Masculine    20
Name: count, dtype: int64

Saved significant interactions to: significant_personal_traits_results/significant_personal_traits_int